Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Master Course #24512

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# SVD Subspace Playground

1. SVD Setup:
- We create made up `U`, `V` and `S` matrices
- We get ground truth / reference matrix `A = U @ S @ V.T`

2. SVD Validation
- We solve the eigenwert problem of `A.T @ A` to get validation versions of `V` and `S`
- We get validation version of `U` from these `V` and `S`
- By that we so far linked the row space and the column space
- We find an orthonormal basis for null space and an orthonormal basis for the left null space
- We create full `U` and `V` matrices containing all four subspaces
- We check if `U` and `V` are unitary
- We check the achieved `A = U @ S @ V.T` with the reference SVD from step 1

If we can do this on our own, we understood most (if not almost all) of the SVD fundamentals and essence. It requires knowlege on
- subspaces
- orthogonal complementary subspaces
- projectors
- creating orthogonal basis
- the eigenwert problem for symmetric matrices (real eigenvalues >=0 / orthogonal eigenvectors)
- the characteristics of the SVD matrices and where these matrices come from in textbook mindset


Once we've managed this, we are probably ready to investigate a real numerical SVD algorithm. This is a scientific playground for itself.

# Big Picture of Four Subspaces / Forward Problem

<img src='slides/four_subspaces.png' alt="drawing" width="1000"/>

In [1]:
import numpy as np

from scipy.linalg import svd, diagsvd, norm, inv, eigh, qr

np.set_printoptions(precision=3, suppress=True)

In [2]:
M, N = 6, 5
R = np.min([M, N]) - 2
M, N, R

(6, 5, np.int64(3))

In [3]:
# create an orthonormal U from reflection
u = np.random.randn(M, 1)
u /= norm(u)
U = np.eye(M) - 2*u@u.T

# create an orthonormal V from reflection
v = np.random.randn(N, 1)
v /= norm(v)
V = np.eye(N) - 2*v@v.T

In [4]:
# check unitary
[np.allclose(U.T@U, np.eye(M)),
 np.allclose(U.T@U, np.eye(M)),
 np.allclose(inv(U), U.T),
 np.allclose(V.T@V, np.eye(N)),
 np.allclose(V.T@V, np.eye(N)),
 np.allclose(inv(V), V.T)]

[True, True, True, True, True, True]

In [5]:
# set up a simple singular value scheme
S = np.zeros((M, N))
S[:R, :R] = np.diag(np.arange(R, 0, -1))
S

array([[3., 0., 0., 0., 0.],
       [0., 2., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]])

In [6]:
A = U @ S @ V.T  # set up reference matrix with SVD
A  # we don't need to care too much about the actual
# random numbers, once we understood the essence

array([[ 1.372,  0.998, -0.265, -0.616, -2.057],
       [ 0.735,  1.545,  0.104,  0.283,  0.945],
       [-0.468, -0.132, -0.527,  0.292,  0.974],
       [ 0.198,  0.041,  0.596, -0.107, -0.358],
       [-0.167, -0.035, -0.503,  0.09 ,  0.302],
       [-0.068, -0.014, -0.206,  0.037,  0.124]])

# Eigenwert Problem for A' A -> Get V and S

In [7]:
De, Ve = eigh(A.T@A)  # symmetric, pos. (semi)-definite
# we sort due to SVD's convention
idx = np.flip(np.argsort(De))
De = De[idx]
Ve = Ve[:, idx]
# we consider only the non-zero eig values
idx = De > 10*np.finfo(float).eps  # tolerance chosen for this toy problem
De = De[idx]
Ve = Ve[:, idx]

## Singular Values

In [8]:
Smf = np.zeros((M, N))
Smf[:R, :R] = np.diag(np.sqrt(De))
Smf  # 'm' stands for my, 'f' for full SVD dimension

array([[3., 0., 0., 0., 0.],
       [0., 2., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]])

# Row Space Span

In [9]:
V_RS = Ve[:, :R]
V_RS  # we don't need to care too much about the actual
# random numbers, once we understood the essence

array([[-0.5  ,  0.341, -0.038],
       [-0.341,  0.767,  0.026],
       [-0.038, -0.026, -0.997],
       [ 0.228,  0.156, -0.018],
       [ 0.762,  0.52 , -0.059]])

In [10]:
V_RS.shape[1], R  # dim of RS == dim of CS

(3, np.int64(3))

# Column Space Span

In [11]:
# dividing by singular values is fine for this toy problem
# dividing by singular values is textbook didactics
# don't do this for real problems
U_CS = A @ V_RS / np.diag(Smf[:R, :R])  
U_CS  # we don't need to care too much about the actual
# random numbers, once we understood the essence

array([[-0.908,  0.038,  0.369],
       [-0.038,  0.984, -0.152],
       [ 0.369,  0.152,  0.478],
       [-0.144, -0.06 , -0.578],
       [ 0.122,  0.05 ,  0.488],
       [ 0.05 ,  0.021,  0.2  ]])

In [12]:
U_CS.shape[1], R  # dim of CS == dim of RS

(3, np.int64(3))

# Left Null Space Span

In [13]:
P_CS = U_CS @ U_CS.T  # projector to CS
P_LNS = np.eye(M) - P_CS  # projector to LNS (orthogonal complement to CS)
tmp = P_LNS @ np.random.randn(M, M-R)  # get some vectors that live in LNS
# but which are not yet orthonormal
tmp, P_CS.T @ tmp  # check

(array([[ 0.224, -0.52 ,  0.253],
        [-0.092,  0.215, -0.104],
        [ 0.895, -2.083,  1.014],
        [ 0.204, -1.233,  1.374],
        [-0.898,  1.413,  0.534],
        [ 0.158, -0.91 , -0.305]]),
 array([[ 0., -0.,  0.],
        [-0.,  0., -0.],
        [-0.,  0., -0.],
        [ 0., -0.,  0.],
        [-0.,  0., -0.],
        [-0.,  0., -0.]]))

In [14]:
# make LNS vectors orthonormal
Qgs, Rgs = qr(tmp)  # by Gram Schmidt idea
Rgs  # use the columns of Q that are weighted by the non-zero rows of R

array([[-1.316,  2.784, -0.551],
       [ 0.   , -1.118,  1.246],
       [ 0.   ,  0.   , -1.231],
       [ 0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ]])

In [15]:
U_LNS = Qgs[:, :(M-R)]
M-R  # dim of LNS

np.int64(3)

In [16]:
Umf = np.hstack((U_CS, U_LNS))  # CS and LNS into full U matrix
Umf.T@U  # check manually created vs. ground truth
# LNS basis is unique
# however the vectors that span LNS, i.e. define the LNS basis, are NOT unique
# hence the potential difference:

array([[-1.   , -0.   ,  0.   ,  0.   , -0.   ,  0.   ],
       [-0.   ,  1.   , -0.   ,  0.   ,  0.   , -0.   ],
       [ 0.   , -0.   , -1.   , -0.   , -0.   , -0.   ],
       [-0.   ,  0.   , -0.   , -0.42 ,  0.907, -0.028],
       [-0.   ,  0.   ,  0.   ,  0.784,  0.379,  0.492],
       [-0.   ,  0.   ,  0.   , -0.457, -0.185,  0.87 ]])

# Null Space Span

In [17]:
P_RS = V_RS @ V_RS.T  # projector to RS
P_NS = np.eye(N) - P_RS  # projector to NS (orthogonal complement to RS)
tmp = P_NS @ np.random.randn(N, N-R)  # get some vectors that live in NS
# but which are not yet orthonormal
tmp, P_RS.T @ tmp  # check

(array([[ 0.391,  0.099],
        [-0.267, -0.068],
        [-0.03 , -0.008],
        [-0.444,  1.076],
        [ 0.268, -0.288]]),
 array([[ 0., -0.],
        [ 0., -0.],
        [ 0.,  0.],
        [ 0., -0.],
        [ 0., -0.]]))

In [18]:
# make NS vectors orthonormal
Qgs, Rgs = qr(tmp)  # by GramSchmidt idea
Rgs  # use the columns of Q that are weighted by the non-zero rows of R

array([[-0.703,  0.708],
       [ 0.   ,  0.869],
       [ 0.   ,  0.   ],
       [ 0.   ,  0.   ],
       [ 0.   ,  0.   ]])

In [19]:
V_NS = Qgs[:, :(N-R)]
N-R  # dim of NS

np.int64(2)

In [20]:
Vmf = np.hstack((V_RS, V_NS))
Vmf.T@V  # check manually created vs. ground truth
# NS basis is unique
# however the vectors that span NS, i.e. define the NS basis, are NOT unique
# hence the potential difference:

array([[-1.   , -0.   ,  0.   ,  0.   ,  0.   ],
       [-0.   ,  1.   , -0.   , -0.   ,  0.   ],
       [ 0.   , -0.   , -1.   , -0.   ,  0.   ],
       [ 0.   , -0.   , -0.   ,  0.885,  0.465],
       [ 0.   , -0.   , -0.   ,  0.465, -0.885]])

# Check my full SVD

In [21]:
np.allclose(Umf @ Smf @ Vmf.T, A)

True

In [22]:
[np.allclose(Umf.T@Umf, np.eye(M)),
 np.allclose(Umf.T@Umf, np.eye(M)),
 np.allclose(inv(Umf), Umf.T),
 np.allclose(Vmf.T@Vmf, np.eye(N)),
 np.allclose(Vmf.T@Vmf, np.eye(N)),
 np.allclose(inv(Vmf), Vmf.T)]


[True, True, True, True, True, True]

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- feel free to use the notebooks for your own purposes
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.